# Pika-style Phone Video Generator - Kaggle version

**Before running anything:**
1. Right sidebar -> **Accelerator** -> pick a GPU (T4 x2 or P100).
2. Right sidebar -> **Internet** -> turn **ON** (needed for git clone / pip
   install / the public link).
3. Don't run cells one by one for the real generation run. Instead use
   **Save & Run All (Commit)** (top-right button). That mode keeps running
   in the background even if you close your phone's browser or lock the
   screen — up to Kaggle's session cap (~9-12 hours), and unlike Colab it
   does not need an open tab to stay alive.
4. After committing, open the run from **your Notebook > Viewer / Logs**
   to find the printed public link (looks like `https://xxxx.gradio.live`)
   and open that link on your phone.

**Kaggle free-tier limits (know these before you start):**
- ~9-12 hours per session, **30 GPU-hours per week** total quota (resets
  weekly). This is *not* 24/7 hosting — it's a much less babysitting-heavy
  version of the same free-tier tradeoff as Colab.
- `/kaggle/working/` is where outputs should go — its contents are saved
  as this notebook Version's output when you commit (downloadable from
  the Output tab), similar to how Drive worked in the Colab version.


In [ ]:
# 1. Confirm GPU is attached
import subprocess
try:
    subprocess.run(['nvidia-smi'], check=True)
except Exception as exc:
    raise RuntimeError(
        'No GPU detected. Open the right sidebar -> Accelerator -> choose a GPU, then rerun.'
    ) from exc


In [ ]:
# 2. Workspace paths (Kaggle's own persistent output folder, no Drive needed)
from pathlib import Path

WAN2GP_ROOT = Path('/kaggle/working/Wan2GP').resolve()
WAN_DATA_ROOT = Path('/kaggle/working/Wan2GP-data').resolve()

for sub in ('ckpts', 'loras', 'outputs', 'cache'):
    (WAN_DATA_ROOT / sub).mkdir(parents=True, exist_ok=True)

WAN_CKPTS_DIR = WAN_DATA_ROOT / 'ckpts'
WAN_LORAS_DIR = WAN_DATA_ROOT / 'loras'
WAN_OUTPUTS_DIR = WAN_DATA_ROOT / 'outputs'
WAN_CACHE_DIR = WAN_DATA_ROOT / 'cache'

print(f'Data root: {WAN_DATA_ROOT}')
print('Note: this folder is only guaranteed to persist for THIS session/commit.')
print('Grab your final MP4 from the notebook Output tab after the run finishes.')


In [ ]:
# 3. Clone Wan2GP and link its data folders to Kaggle working storage
import subprocess, shutil

if not WAN2GP_ROOT.exists():
    subprocess.run(['git', 'clone', 'https://github.com/deepbeepmeep/Wan2GP.git', str(WAN2GP_ROOT)], check=True)
else:
    print('Wan2GP already present in this session.')

def link(repo_sub, data_dir):
    target = WAN2GP_ROOT / repo_sub
    if target.is_symlink():
        return
    if target.exists():
        for item in target.iterdir():
            shutil.move(str(item), str(data_dir / item.name))
        target.rmdir()
    target.symlink_to(data_dir, target_is_directory=True)

link('ckpts', WAN_CKPTS_DIR)
link('loras', WAN_LORAS_DIR)
link('outputs', WAN_OUTPUTS_DIR)
print('Linked.')


In [ ]:
# 4. System dependencies (ffmpeg + libs)
import subprocess
subprocess.run(['sudo', 'apt-get', '-qq', 'update'], check=True)
subprocess.run(['sudo', 'apt-get', '-qq', 'install', '-y', 'ffmpeg', 'libglib2.0-0', 'libgl1', 'libportaudio2'], check=True)
print('System dependencies installed.')


In [ ]:
# 5. Python dependencies (PyTorch + Wan2GP + this repo's orchestrator)
import subprocess, sys, os

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
                'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
                '--index-url', 'https://download.pytorch.org/whl/cu128'], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                str(WAN2GP_ROOT / 'requirements.txt')], check=True)

REPO_DIR = '/kaggle/working/Hika'
if not os.path.exists(REPO_DIR):
    subprocess.run(['git', 'clone', 'https://github.com/mrtuik/Hika.git', REPO_DIR], check=True)

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', '-r',
                f'{REPO_DIR}/requirements.txt'], check=True)
print('Dependencies installed.')


In [ ]:
# 5b. Headless matplotlib fix (needed for Wan2GP's preprocessing tools)
target = WAN2GP_ROOT / 'preprocessing/matanyone/tools/interact_tools.py'
if target.exists():
    text = target.read_text()
    if "matplotlib.use('TkAgg')" in text:
        target.write_text(text.replace("matplotlib.use('TkAgg')", "matplotlib.use('Agg')", 1))
        print('Patched matplotlib backend to Agg.')
    else:
        print('No patch needed.')
else:
    print('File not found - skipping (Wan2GP version may differ).')


In [ ]:
# 7. Launch Wan2GP headless (LOCAL ONLY - our own UI below is the public one)
import subprocess, os, time

env = os.environ.copy()
env['HF_HOME'] = str(WAN_CACHE_DIR / 'huggingface')

wan2gp_process = subprocess.Popen(
    ['python', 'wgp.py', '--listen', '--server-port', '7860', '--profile', '5'],
    cwd=str(WAN2GP_ROOT),
    env=env,
)
print('Starting Wan2GP on http://127.0.0.1:7860 ... waiting 40s')
time.sleep(40)
print('Wan2GP should be up now.')


In [ ]:
# 8. Inspect Wan2GP's API (run once, confirm endpoint name)
import sys
sys.path.insert(0, '/kaggle/working/Hika/src')
from wan2gp_client import inspect_api
inspect_api()
# Match what's printed here against WAN2GP_API_NAME in src/wan2gp_client.py
# and adjust that file if they differ.


## 9. What to expect

One story + one character photo produces **one Wan2GP video clip** (no multi-scene splitting, no voice, no lip-sync). Measured on a free T4-class GPU: **~8 minutes for a 5-second 480p clip**.


In [ ]:
# 10. Launch your custom Pika-style UI
# This prints a public https://xxxx.gradio.live link in the log/output -
# that is the link to open on your phone, including after you close this
# browser tab (as long as you used "Save & Run All / Commit" to start this
# run, not an interactive session).
import os
os.chdir('/kaggle/working/Hika/src')
os.system('python orchestrator_ui.py')
